# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AnanyaGujral/flyrank-ml-internship-start/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook runs **offline** on the bundled anonymized slice (`data/raw/content_refresh_anonymized.csv`, 30,000 pages, 32 pseudonymized clients) — no warehouse token needed.

## 1. My lane (or freestyle) and why

**Lane: Refresh / Content Opportunity Scoring (Lane 2).**

A content portfolio holds thousands of pages; an editor can review only a few hundred a week. The
useful question is not "which pages exist" but "which already-visible pages are drifting, and in what
order should a human look at them". That is an *opportunity-scoring / ranking* problem, and it is the
one place where a small, honest model can pay for itself — if it beats the transparent alternative.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

root = Path.cwd().resolve()
while root != root.parent and not (root / "AGENTS.md").exists():
    root = root.parent
raw = pd.read_csv(root / "data" / "raw" / "content_refresh_anonymized.csv")
pool = raw[raw["impressions_prev_30d"] >= 100].copy()
pool["declined_next_30d"] = (pool["impressions_last_30d"] < 0.8 * pool["impressions_prev_30d"]).astype(int)
print(f"{len(raw):,} pages | {raw['client_id'].nunique()} pseudo-clients | {raw.shape[1]} columns")
print(f"already-visible pool (prior-30 impressions >= 100): {len(pool):,} pages "
      f"across {pool['client_id'].nunique()} clients")
print(f"decline base rate in that pool: {pool['declined_next_30d'].mean():.3f}")

30,000 pages | 32 pseudo-clients | 44 columns
already-visible pool (prior-30 impressions >= 100): 18,010 pages across 30 clients
decline base rate in that pool: 0.616


## 2. The question: decision, action, cost of a wrong call

**Decision this improves:** which of a client's already-visible pages an editor reviews first.

**Who acts, and what do they do:** a FlyRank content strategist/editor works down a ranked queue —
refresh, expand, protect, prune, or simply monitor — one human decision per page.

**Cost of a wrong call:** a false alarm burns review time on a healthy page; a miss leaves a page that
is already losing demand unowned. Both are cheap per row and expensive at portfolio scale, which is why
the queue is ranked rather than auto-actioned.

**Research question.** *Among a client portfolio's already-visible pages, can trailing search and
engagement signals rank the pages that are about to decline well enough to beat a transparent rule —
measured at the top of the queue, under a client-grouped (new-client) validation design?*

In [2]:
# How big is the "worth reviewing" candidate pool, and how much demand does it cover?
share_impressions = pool["impressions_prev_30d"].sum() / raw["impressions_prev_30d"].sum()
print(f"candidate pool: {len(pool):,} of {len(raw):,} pages ({100*len(pool)/len(raw):.1f}%)")
print(f"they carry {100*share_impressions:.1f}% of all prior-30 impressions in the slice")
print(f"declining share inside the pool: {pool['declined_next_30d'].mean():.3f}")

candidate pool: 18,010 of 30,000 pages (60.0%)
they carry 99.5% of all prior-30 impressions in the slice
declining share inside the pool: 0.616


## 3. Quick look at the data (2-3 real numbers)

Three numbers that make the lane worth the effort: how common decline is, how big the visible pool is,
and how skewed the review burden would be if every page were treated equally.

In [3]:
declining = pool[pool["declined_next_30d"] == 1]
print(f"1) decline is the MAJORITY outcome in the visible pool: {len(declining):,} pages "
      f"({100*pool['declined_next_30d'].mean():.1f}%) lost >20% of impressions window-over-window")
print(f"2) the pool is small enough to rank but too big to read: {len(pool):,} pages "
      f"= ~{len(pool)/300:.0f} weeks of 300-page review sessions")
print(f"3) exposure is heavy-tailed: the top 10% of pool pages carry "
      f"{100*pool['impressions_prev_30d'].nlargest(int(0.1*len(pool))).sum()/pool['impressions_prev_30d'].sum():.0f}% "
      "of pool demand")
print("\n=> a good ranking, not a bigger list, is what saves editorial time.")

1) decline is the MAJORITY outcome in the visible pool: 11,090 pages (61.6%) lost >20% of impressions window-over-window
2) the pool is small enough to rank but too big to read: 18,010 pages = ~60 weeks of 300-page review sessions
3) exposure is heavy-tailed: the top 10% of pool pages carry 61% of pool demand

=> a good ranking, not a bigger list, is what saves editorial time.


## 4. Careful words: what I can and can't claim

**What this work can say:**
- *observed / measured:* how well a transparent rule and a learned model rank next-window decline on
  this snapshot, under a client-grouped split.
- *directional:* which trailing signals are associated with decline.
- *decision-support:* a ranked, human-reviewed queue.

**What this work cannot say, and won't claim:**
- that refreshing a page *causes* recovery (that needs a controlled experiment);
- that any of this explains or predicts Google's ranking algorithm — the data only holds observed
  impressions, clicks, position, and engagement;
- any client-identifying detail: identifiers are pseudonyms and are used for grouping only.

In [4]:
claim_ladder = {
    "observed":   "decline is the majority outcome in the visible pool",
    "measured":   "rule vs model ranking quality on held-out clients",
    "directional":"which trailing signals associate with decline",
    "decision-support": "a ranked queue a human reviews",
    "NOT CLAIMED": "causal refresh effect; Google's algorithm; per-client detail",
}
for kind, what in claim_ladder.items():
    print(f"[{kind:>16}] {what}")

[        observed] decline is the majority outcome in the visible pool
[        measured] rule vs model ranking quality on held-out clients
[     directional] which trailing signals associate with decline
[decision-support] a ranked queue a human reviews
[     NOT CLAIMED] causal refresh effect; Google's algorithm; per-client detail


## Self-check

- [x] Lane named, question written as a decision + metric
- [x] Runs top to bottom offline on the bundled slice
- [x] Only careful words (observed / measured / directional / decision-support)
- [x] No client names, domains, URLs, or queries — pseudonymous ids only
- [x] Committed under `work/notebooks/`